In [ ]:
%load_ext autoreload
%autoreload 2

# 1. Imports and example data

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from pywatemsedem.catchment import Catchment
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.io.modelinput import Modelinput
from pywatemsedem.io.modeloutput import Modeloutput
from pywatemsedem.postprocess import PostProcess
from pywatemsedem.scenario import Scenario

In [ ]:
# Example data folder
inputdata_folder = Path("../../tests/data")
analysis_folder = Path("/nas/fluves/projecten/CP2022.011_Departement_Omgeving_CNWS_2/02_results/simplified_routing")
results_folder = analysis_folder / "runs"

# WaTEM-SEDEM binary
os.environ["WATEMSEDEM"] = os.getcwd()
watem_sedem_folder = Path(os.environ.get("WATEMSEDEM"))
watem_sedem_binary_regular = analysis_folder / "binaries" / "20260817_watem_sedem_regular"
watem_sedem_binary_simplified_routing = analysis_folder / "binaries" / "20260819_watem_sedem_simplified_routing"

# 2. Model calculations

## 2.1. Catchment object

In [ ]:
name_catchment = 'langegracht'

rst_dtm = inputdata_folder  / "dtm.tif"
vct_catchment = inputdata_folder / "catchm_langegracht.shp"

catchment = Catchment(name_catchment, vct_catchment, rst_dtm, 20, 31370, -9999, results_folder=results_folder)

rst_kfactor = inputdata_folder / "K3.tif"
catchment.kfactor = rst_kfactor

rst_landuse = inputdata_folder / "basemap_landuse.tif"
catchment.landuse = rst_landuse

vct_river = inputdata_folder / "river.shp"
catchment.vct_river = vct_river

vct_infra_poly = inputdata_folder / "infrastructure.shp"
catchment.vct_infrastructure_buildings = vct_infra_poly

vct_infra_line = inputdata_folder / "roads.shp"
catchment.vct_infrastructure_roads = vct_infra_line

vct_water = inputdata_folder / "water.shp"
catchment.vct_water = vct_water

## 2.2. User choices object

In [ ]:
default_choices = inputdata_folder / "userchoices.ini"

options = Options()
options.read_values_from_ini(default_choices)

parameters = Parameters()
parameters.read_values_from_ini(default_choices)

extensions = Extensions()
extensions.read_values_from_ini(default_choices)

output = Output()
output.read_values_from_ini(default_choices)

extensionparameters = ExtensionsParameters(extensions)
extensionparameters.apply_defaults()
extensionparameters.ktc_low.read_value_from_ini(default_choices)
extensionparameters.ktc_high.read_value_from_ini(default_choices)
extensionparameters.ktc_limit.read_value_from_ini(default_choices)
extensionparameters.sewer_exit.read_value_from_ini(default_choices)

choices = Choices(options, parameters, extensions, extensionparameters, output)

## 2.3. Scenario calculations

### 2.3.1. Scenario 1: regular routing

In [ ]:
# Scenario object 
scenario_nr = 1
scenario = Scenario(catchment, 2019, scenario_nr, choices)

scenario.vct_parcels = inputdata_folder / r"parcels.shp"
scenario.vct_parcels.geodata

scenario.choices.extensions.include_buffers = True
scenario.vct_buffers = inputdata_folder / r"buffers.shp"

scenario.choices.extensions.include_grass_strips = True
scenario.vct_grass_strips = inputdata_folder / r"grass_strips.shp"

scenario.choices.extensions.include_sewers = True
scenario.vct_endpoints = inputdata_folder / r"endpoints.shp"

scenario.composite_landuse = scenario.create_composite_landuse()

scenario.cfactor = scenario.create_cfactor()

scenario.ktc = scenario.create_ktc(choices.extensionparameters.ktc_low.value,
                                   choices.extensionparameters.ktc_high.value,
                                   choices.extensionparameters.ktc_limit.value,
                                   not choices.extensions.create_ktc_map.value)

scenario.prepare_input_files()
scenario.create_ini_file()

# Run model
scenario.run_model(watem_sedem_binary_regular)

scenario_regular = scenario
modelinput_regular = Modelinput(scenario_regular.ini, 31370)
modeloutput_regular = Modeloutput(scenario_regular.ini, 31370)

### 2.3.2. Scenario 2: simplified routing

In [ ]:
# Scenario object 
scenario_nr = 2
scenario = Scenario(catchment, 2019, scenario_nr, choices)

scenario.vct_parcels = inputdata_folder / r"parcels.shp"
scenario.vct_parcels.geodata

scenario.choices.extensions.include_buffers = True
scenario.vct_buffers = inputdata_folder / r"buffers.shp"

scenario.choices.extensions.include_grass_strips = True
scenario.vct_grass_strips = inputdata_folder / r"grass_strips.shp"

scenario.choices.extensions.include_sewers = True
scenario.vct_endpoints = inputdata_folder / r"endpoints.shp"

scenario.composite_landuse = scenario.create_composite_landuse()

scenario.cfactor = scenario.create_cfactor()

scenario.ktc = scenario.create_ktc(choices.extensionparameters.ktc_low.value,
                                   choices.extensionparameters.ktc_high.value,
                                   choices.extensionparameters.ktc_limit.value,
                                   not choices.extensions.create_ktc_map.value)

scenario.prepare_input_files()
scenario.create_ini_file()

# Run model
scenario.run_model(watem_sedem_binary_simplified_routing)

scenario_simplified = scenario
modelinput_simplified = Modelinput(scenario_simplified.ini, 31370)
modeloutput_simplified = Modeloutput(scenario_simplified.ini, 31370)

# 3. Comparative analysis

In [ ]:
import routing_comparison as rc

routing_scenarios = {
    "regular": {
        "scenario": scenario_regular,
        "modelinput": modelinput_regular,
        "modeloutput": modeloutput_regular,
    },
    "simplified": {
        "scenario": scenario_simplified,
        "modelinput": modelinput_simplified,
        "modeloutput": modeloutput_simplified,
    },
}

catchment_totals = rc.compare_scenarios(
    pd.DataFrame(
        {
            scenario_name: rc.total_sediment_summary(objects["modeloutput"])
            for scenario_name, objects in routing_scenarios.items()
        }
    )
)
catchment_totals.index.name = "metric"

landuse_totals = pd.concat(
    [
        rc.sediment_by_landuse(objects["modelinput"], objects["modeloutput"]).assign(
            scenario=scenario_name
        )
        for scenario_name, objects in routing_scenarios.items()
    ],
    ignore_index=True,
)

landuse_comparison = (
    landuse_totals
    .pivot_table(
        index=["sediment_variable", "landuse_group"],
        columns="scenario",
        values="sediment_kg",
        aggfunc="sum",
    )
    .pipe(rc.compare_scenarios)
    .reset_index()
)
landuse_comparison.columns.name = None

grass_strip_totals = pd.concat(
    [
        rc.postprocess_grass_strips(objects["scenario"], objects["modeloutput"]).assign(
            scenario=scenario_name
        )
        for scenario_name, objects in routing_scenarios.items()
    ],
    ignore_index=True,
)

grass_strip_summary_metrics = [
    column
    for column in ["sedi_in", "sedi_out", "sed"]
    if column in grass_strip_totals.columns
]

grass_strip_summary = (
    grass_strip_totals
    .groupby("scenario")[grass_strip_summary_metrics]
    .sum()
    .transpose()
)

if {"sedi_in", "sedi_out"}.issubset(grass_strip_summary.index):
    grass_strip_summary.loc["eSTE"] = (
        1 - grass_strip_summary.loc["sedi_out"] / grass_strip_summary.loc["sedi_in"]
    ) * 100

grass_strip_summary = rc.compare_scenarios(grass_strip_summary)
grass_strip_summary.index.name = "metric"

rc.display_table("Catchment totals", catchment_totals)
rc.display_table("Sediment by landuse", landuse_comparison)
rc.display_table("Grass strips summary", grass_strip_summary)

### Verschil tussen *Grass strips summary* en de andere tabellen

In *Catchment totals* en *Sediment by landuse* zijn `sedi_in` en `sedi_out` sommen over pixels (over het hele bekken, resp. per landgebruiksklasse). Sediment dat van pixel naar pixel stroomt, wordt daardoor in elke gepasseerde pixel opnieuw meegeteld (als `sedi_out` van de ene pixel én als `sedi_in` van de volgende).

*Grass strips summary* telt enkel het sediment dat de rand van een grasbufferstrook overschrijdt. Het intern transport binnen de grasbufferstrook wordt niet meegeteld, zodat `sedi_in` en `sedi_out` de werkelijke in- en uitstroom per grasbufferstrook zijn. Daarom liggen beide waarden lager dan de grasbufferstrooklijnen in *Sediment by landuse*; het verschil is precies het interne transport en is voor `sedi_in` en `sedi_out` gelijk.

### Sedimenttransport tussen landgebruiksklassen

Om te begrijpen waar de verschillen vandaan komen, wordt het sediment per routingpijl gevolgd: elke pijl draagt de `sedi_out` van zijn bronpixel vermenigvuldigd met zijn `part`. Door te groeperen op het landgebruik van bron- en doelpixel wordt duidelijk:

- de belangrijkste sedimentstromen tussen verschillende landgebruiksklassen (> 50 000 kg in minstens één scenario), gesorteerd op het grootste absolute verschil;
- voor infrastructuur: het interne transport van wegpixel naar wegpixel, de in- en uitstroom, het aantal wegpixels dat het sediment passeert vóór het de weg verlaat (langs de hoofdrichting van elke pixel) en het gemiddelde hoogteverschil per stap;
- voor de grasbufferstroken: de instroom per herkomst;
- voor de vereenvoudigde routing: welke routingbeslissingen genomen werden voor de wegpixels (uit `routing_detailed.txt`, geschreven door de binary van 20260819).

In [ ]:
routing_arrows = {
    scenario_name: rc.routing_arrows(objects["modelinput"], objects["modeloutput"])
    for scenario_name, objects in routing_scenarios.items()
}

# Sediment transport between land-use groups
flows_between_groups = rc.compare_scenarios(
    pd.DataFrame(
        {
            scenario_name: arrows.groupby(["group_source", "group_target"])["sediment_kg"].sum()
            for scenario_name, arrows in routing_arrows.items()
        }
    ).fillna(0.0)
)
flows_between_groups.index.names = ["van", "naar"]

# Only show the main flows between different land-use groups (internal transport within a
# group is shown for infrastructure below), sorted on the largest absolute difference
van = flows_between_groups.index.get_level_values("van")
naar = flows_between_groups.index.get_level_values("naar")
main_flows_between_groups = (
    flows_between_groups.loc[
        (van != naar)
        & (flows_between_groups[["regular", "simplified"]].max(axis=1) > 50000)
    ]
    .sort_values("absolute_difference", key=abs, ascending=False)
)

# Infrastructure: internal transport vs. inflow, path length and elevation drop per step
infrastructure_rows = {}
for scenario_name, arrows in routing_arrows.items():
    infra_source = arrows["group_source"] == "infrastructure"
    infra_target = arrows["group_target"] == "infrastructure"
    path_length = rc.infrastructure_path_length(arrows)
    infrastructure_rows[scenario_name] = {
        "wegpixel -> wegpixel (kg)": arrows.loc[infra_source & infra_target, "sediment_kg"].sum(),
        "instroom op wegen (kg)": arrows.loc[~infra_source & infra_target, "sediment_kg"].sum(),
        "uitstroom van wegen (kg)": arrows.loc[infra_source & ~infra_target, "sediment_kg"].sum(),
        "gepasseerde wegpixels (gemiddeld)": path_length.mean(),
        "gepasseerde wegpixels (p90)": np.percentile(path_length, 90),
        "hoogteverschil per stap wegpixel -> wegpixel (m)": arrows.loc[
            infra_source & infra_target, "elevation_drop"
        ].mean(),
    }
infrastructure_summary = rc.compare_scenarios(pd.DataFrame(infrastructure_rows))
infrastructure_summary.index.name = "metric"

# Grass strips: inflow per source land-use group
grass_strip_inflow = flows_between_groups.xs("grass strips", level="naar").drop(
    index="grass strips", errors="ignore"
)

rc.display_table("Belangrijkste sedimentstromen tussen landgebruiksklassen", main_flows_between_groups)
rc.display_table("Infrastructuur: intern transport, instroom en padlengte", infrastructure_summary)
rc.display_table("Grasbufferstroken: instroom per herkomst", grass_strip_inflow)

# Routing decisions for infrastructure pixels (detailed routing output of the simplified routing binary)
routing_detailed = pd.read_table(
    modeloutput_simplified.modeloutputfolder / "routing_detailed.txt"
)
landuse = np.asarray(modelinput_simplified.compositelanduse.arr)
is_infra = landuse[routing_detailed["row"] - 1, routing_detailed["col"] - 1] == -2
infrastructure_decisions = (
    routing_detailed.loc[is_infra]
    .fillna("-")
    .groupby(["decision_candidate", "decision_find_lower"])
    .size()
    .rename("aantal wegpixels")
    .to_frame()
)
rc.display_table("Routingbeslissingen wegpixels (simplified)", infrastructure_decisions)

## 4. Belangrijkste conclusies

Testbekken Langegracht; reguliere vs. vereenvoudigde routing.

- **De vereenvoudigde routing geeft consequent lagere sedimentfluxen.** Alle bekkentotalen dalen, geen enkel stijgt: sedimentexport naar de rivier −9 %, rioleringen −11,5 %, sinks −9,5 %.
- **De over de pixels gesommeerde `sedi_in`/`sedi_out` dalen sterker (−13 %) dan de export (−9 %).** Deze sommen tellen het sediment in elke pixel die het passeert, wat wijst op kortere stroompaden (minder gepasseerde pixels) en niet enkel op minder sediment.
- **Infrastructuur verklaart het grootste deel van het verschil.** Het sediment in/uit op infrastructuur daalt met −25 % (≈1,9 M kg), de grootste relatieve wijziging en ongeveer een derde van het totale absolute verschil. De vereenvoudigde routing verandert vooral hoe sediment over wegen en gebouwen wordt gerouteerd.
  - Bijna de volledige daling is transport van wegpixel naar wegpixel (6,65 → 4,79 M kg); de instroom op de wegen daalt slechts met −7 %. Het sediment verlaat de weg sneller (gemiddeld 14,5 → 12,4 wegpixels), omdat `FindLower` nu de laagste buur kiest (steilste afdaling) in plaats van de aspect-buur.
- **Percelen volgen de trend van het bekken (−10 tot −11 %)**, terwijl bos praktisch ongewijzigd blijft (< 1 %) en grasland matig wijzigt (−4 tot −7 %).
- **Relatief bereikt meer sediment de grasbufferstroken.** De instroom in de grasbufferstroken daalt slechts met −3 %, tegenover −9 à −12 % voor de andere eindbestemmingen (rivier, rioleringen, sinks). Het aandeel sediment dat in de grasbufferstroken terechtkomt neemt dus toe (bv. verhouding instroom grasbufferstroken/rivierexport 4,11 → 4,38).
  - De instroom vanuit percelen blijft gelijk (2,41 M kg); de daling komt vooral van minder instroom vanaf wegen (132 k → 78 k kg), omdat de uitzondering voor grasbufferstroken nu enkel geldt voor perceelpixels.
- **De efficiëntie van de grasbufferstroken stijgt licht.** Sediment naar de grasbufferstroken −3 %, sediment eruit −11 %, eSTE 85,2 % → 86,4 % (+1,2 procentpunt).